# AgroPest-12 Part B: DINOv3-Guided YOLOv12m Pretraining

**Student:** Random YOLOv12m backbone  
**Teacher:** Frozen released DINOv3 ViT-B/16  
**Dataset:** AgroPest-12-PartB-Safe SSL pool (9,180 images)  
**Platform:** Kaggle T4 x2  

This notebook performs strict label-free, DINOv3-guided feature distillation. It reads only `ssl_pool/images`; labels, validation images, and test images are never accessed.

## Method and Protocol

A released DINOv3 ViT-B/16 is kept frozen as a feature teacher. A randomly initialized YOLOv12m CNN student sees the same augmented image and learns to match both the teacher's global class-token embedding and its dense patch-token feature map. The global and dense objectives are independent cosine regressions: \(D(a,b)=2-2\cos(a,b)\).

The student encoder produces a spatial feature map. Its global average-pooled representation is projected to the teacher embedding dimension for global regression; a 1x1 convolution projects the dense map and bilinear interpolation aligns it to the teacher's 14x14 patch grid for dense regression. The frozen teacher, projectors, and optimizer state are not transferred. Only the trained YOLOv12m backbone transfers to detection.

This is **DINOv3-guided domain-adaptive feature distillation**, not full DINOv3 pretraining. Meta's DINOv3 recipe includes DINO self-distillation, iBOT, KoLeo regularization, Gram anchoring, multi-crop views, and large-scale FSDP training. Here, a released DINOv3 teacher is a stationary feature oracle while the YOLO CNN student learns its AgroPest-domain representations without labels.

The Part B partition policy forbids reading labels, validation images, or test images in this notebook. Diagnostics therefore use a fixed image-only sample from the SSL pool and do not make label-based claims.

In [ ]:
# Install the current course SSL library and visualization dependencies.
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "ultralytics>=8.4.96", "scikit-learn", "seaborn"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps", "git+https://github.com/rifat963/ssl-detection-lab.git@main"])

In [ ]:
# ============================================================
# CONFIG -- every value is fixed and viva-explainable
# ============================================================
from pathlib import Path

PARTB_ROOT = Path("/kaggle/input/datasets/sadikshahriar/agropest-12-partb-safe/AgroPest-12-PartB-Safe")
SSL_IMAGES = PARTB_ROOT / "ssl_pool" / "images"
OUTPUT_DIR = Path("/kaggle/working/partB_04a_dinov3_yolo12m")

# Attach the authorized DINOv3 ViT-B/16 LVD-1689M checkpoint as a Kaggle input.
DINOV3_MODEL = "dinov3_vitb16"
DINOV3_WEIGHT_FILENAME = "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
DINOV3_WEIGHT_CANDIDATES = [
    Path("/kaggle/input/dinov3-weigths") / DINOV3_WEIGHT_FILENAME,
    Path("/kaggle/input/dinov3-weights") / DINOV3_WEIGHT_FILENAME,
    Path("/kaggle/input/datasets/mrifatrashid/dinov3-weigths") / DINOV3_WEIGHT_FILENAME,
]
# Kaggle can expose attached datasets beneath different slugs. Search only for
# the exact authorized checkpoint filename; image labels are never opened or read.
DINOV3_DISCOVERED_WEIGHTS = sorted(Path("/kaggle/input").rglob(DINOV3_WEIGHT_FILENAME))
DINOV3_WEIGHTS = next(
    (path for path in [*DINOV3_WEIGHT_CANDIDATES, *DINOV3_DISCOVERED_WEIGHTS] if path.is_file()),
    None,
)
DINOV3_REPOSITORY = "facebookresearch/dinov3"
DINOV3_SOURCE = "github"

SEED = 445                 # Shared Part B seed and fixed diagnostic sample.
YOLO_MODEL = "yolo12m.yaml" # Random CNN student: no COCO-supervised initialization.
EPOCHS = 30                # Assignment B standard SSL pretraining budget.
IMAGE_SIZE = 224           # Course-tested DINOv3 ViT-B/16 T4 setting; 14x14 teacher patch grid retains large-pest morphology while fitting a single session.
PER_GPU_BATCH = 4          # Course DINOv3 T4 reference profile. Frozen ViT-B/16 + YOLOv12m backward pass is substantially heavier than the other SSL methods.
GRAD_ACCUM_STEPS = 4       # Effective global batch = 4 images/GPU x 2 GPUs x 4 accumulation = 32; reduces gradient variance without changing a non-contrastive objective.
WORKERS = 2                # Matches the Part A two-workers-per-GPU data-loading budget.
LEARNING_RATE = 3e-4       # Course-library AdamW rate for random YOLO student regression to a frozen teacher.
MIN_LEARNING_RATE = 3e-6   # 100x lower cosine-decay floor for this 30-epoch run.
WEIGHT_DECAY = 1e-4        # Mild regularization: stronger decay would compete with teacher-feature matching.
WARMUP_EPOCHS = 1          # Frozen teacher targets are stable from step one; one epoch prevents large random-student updates at startup.
GRADIENT_CLIP = 5.0        # Limits early random-student gradient spikes.
GLOBAL_LOSS_WEIGHT = 1.0   # Balanced global semantic cosine regression; the validated library reference uses 1.0.
DENSE_LOSS_WEIGHT = 1.0    # Balanced dense spatial cosine regression; no published YOLO-specific evidence justifies reweighting it.
DIAGNOSTIC_IMAGES = 500    # Fixed image-only SSL-pool sample for t-SNE/retrieval.
RETRIEVAL_QUERIES = 4      # Four fixed query panels.
RETRIEVAL_NEIGHBORS = 5    # Assignment B requires query plus top-five cosine neighbours.

CLASS_NAMES = ["Ants", "Bees", "Beetles", "Caterpillars", "Earthworms", "Earwigs", "Grasshoppers", "Moths", "Slugs", "Snails", "Wasps", "Weevils"]
print(f"CONFIG loaded | DINOv3 ViT-B/16 -> {YOLO_MODEL} | {EPOCHS} epochs | {PER_GPU_BATCH} images/GPU x {GRAD_ACCUM_STEPS} accumulation | {IMAGE_SIZE}px")

## Dataset-Specific CONFIG Rationale

**Teacher:** ViT-B/16 is the strongest released teacher that remains practical on two T4 GPUs. It supplies 768-dimensional global and dense features, compared with 384 dimensions for ViT-S/16. This richer dense supervision is valuable for later object detection. The teacher is frozen, so its forward pass has no activation-gradient memory; however, it still makes `batch_size=4` the safe course-library T4 profile when paired with a random YOLOv12m student.

**Resolution:** 224px is divisible by ViT-B/16's patch size and gives a 14x14 DINOv3 patch grid. This is the current course reference setting and is intentionally retained for reliable dual-T4 runtime. AgroPest's 88.9% large boxes retain sufficient pest morphology at 224px, while the downstream detector still fine-tunes at 640px for localization. Using 256px would make a 16x16 teacher grid but increases frozen-teacher attention cost and is not justified without a matched downstream ablation.

**Batch and accumulation:** This is cosine feature regression, not contrastive learning: there are no in-batch negatives. Gradient accumulation does not change the objective; it makes the optimizer see an effective 32-image global batch and smooths the update. `4/GPU x 2 GPUs x 4` is the course-tested profile that protects the single-session budget while accommodating a ViT-B/16 teacher and YOLOv12m student.

**Balanced objectives:** The implementation computes one mean cosine-regression scalar for global semantics and one for dense spatial features, each bounded from 0 to 4. Equal 1.0 weights are the library reference and are the only non-speculative choice. Dense-weight emphasis might plausibly help localization, but it has not been demonstrated for this DINOv3-to-YOLO objective; tuning it would require a controlled downstream ablation.

**Augmentation:** DINOv3 distillation uses one shared view: random resized crop at scale 0.50-1.0, horizontal flip, and ImageNet normalization. The frozen teacher and student receive exactly the same crop. Unlike SimCLR, two independent views, aggressive colour jitter, grayscale, and blur are not needed because the loss directly regresses corresponding teacher and student features. The 0.50 lower crop bound preserves enough leaf and pest context for large-object AgroPest images.

**Optimization:** A random CNN starts with uncorrelated features relative to the frozen teacher. AdamW at 3e-4, one warmup epoch, cosine decay to 3e-6, and gradient clipping at 5.0 are the library's stable DINOv3-distillation defaults. The teacher's fixed features make a longer warmup unnecessary.

In [ ]:
# ============================================================
# Environment and strict SSL-pool-only checks
# ============================================================
import json, random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO

import ssldet
from ssldet import PretrainConfig, assert_supported_runtime, available_ssl_modules, launch_distributed_pretrain, transfer_ssl_backbone_to_yolo
from ssldet.backbones import DINOV3_SPECS, YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform
from ssldet.registry import build_method

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid")

assert SSL_IMAGES.is_dir(), f"Attach AgroPest-12-PartB-Safe. Missing: {SSL_IMAGES}"
assert not any(SSL_IMAGES.parent.rglob("*.txt")), "SSL pool must not contain label files."
if DINOV3_WEIGHTS is None:
    mounted_inputs = sorted(path.name for path in Path("/kaggle/input").iterdir())
    checked_paths = "\n".join(f"  - {path}" for path in DINOV3_WEIGHT_CANDIDATES)
    raise FileNotFoundError(
        "Attach the DINOv3 ViT-B/16 weights Kaggle dataset in the notebook's Add Input panel.\n"
        "The AgroPest dataset alone does not include this licensed teacher checkpoint.\n"
        f"Expected filename: {DINOV3_WEIGHT_FILENAME}\nChecked:\n{checked_paths}\n"
        f"Mounted inputs: {mounted_inputs}"
    )
assert DINOV3_MODEL in DINOV3_SPECS, f"Unsupported teacher: {DINOV3_MODEL}"
assert IMAGE_SIZE % DINOV3_SPECS[DINOV3_MODEL].patch_size == 0
runtime = assert_supported_runtime(require_cuda=True, minimum_gpus=2)
assert "dinov3" in available_ssl_modules()

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
ssl_paths = sorted(path for path in SSL_IMAGES.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
assert len(ssl_paths) == 9180, f"Expected 9,180 SSL images, found {len(ssl_paths)}"

pd.Series({
    "ssl-detection-lab": ssldet.__version__, "PyTorch": torch.__version__,
    "Ultralytics": runtime["packages"]["ultralytics"]["installed"],
    "GPU count": runtime["cuda"]["device_count"],
    "GPUs": ", ".join(device["name"] for device in runtime["cuda"]["devices"]),
    "SSL images": len(ssl_paths), "label files visible": 0,
    "teacher": DINOV3_SPECS[DINOV3_MODEL].architecture,
    "teacher embedding dim": DINOV3_SPECS[DINOV3_MODEL].embedding_dim,
    "teacher checkpoint MiB": round(DINOV3_WEIGHTS.stat().st_size / 1024**2, 1),
})

## Shared-View Preview

The student and frozen teacher receive the same single augmented image. This is direct feature regression, not a multi-view contrastive objective. Check that the 0.50-1.0 crop keeps enough pest and surrounding leaf context before training.

In [ ]:
config = PretrainConfig(
    method="dinov3", image_roots=[str(SSL_IMAGES)], output_dir=str(OUTPUT_DIR), yolo_model=YOLO_MODEL,
    epochs=EPOCHS, batch_size=PER_GPU_BATCH, image_size=IMAGE_SIZE, workers=WORKERS,
    seed=SEED, learning_rate=LEARNING_RATE, min_learning_rate=MIN_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, grad_accum_steps=GRAD_ACCUM_STEPS,
    gradient_clip=GRADIENT_CLIP, amp=True,
    dinov3_model=DINOV3_MODEL, dinov3_weights=str(DINOV3_WEIGHTS),
    dinov3_repository=DINOV3_REPOSITORY, dinov3_source=DINOV3_SOURCE,
    dinov3_global_weight=GLOBAL_LOSS_WEIGHT, dinov3_dense_weight=DENSE_LOSS_WEIGHT,
    save_every=1,
).validate()

preview_dataset = UnlabeledImageDataset(ssl_paths, build_transform(config))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for axis, index in zip(axes.flat, random.Random(233).sample(range(len(preview_dataset)), 4)):
    image = preview_dataset[index]
    axis.imshow((image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0))
    axis.set_title(ssl_paths[index].name[:32], fontsize=8)
    axis.axis("off")
plt.suptitle("Shared teacher-student input: crop + horizontal flip", y=0.92)
plt.tight_layout(); plt.show()

pd.Series({
    "student initialization": "random yolo12m.yaml",
    "source images per DDP step": PER_GPU_BATCH * runtime["cuda"]["device_count"],
    "effective global batch": PER_GPU_BATCH * runtime["cuda"]["device_count"] * GRAD_ACCUM_STEPS,
    "teacher patch grid": f"{IMAGE_SIZE // 16}x{IMAGE_SIZE // 16}",
    "global:dense weights": f"{GLOBAL_LOSS_WEIGHT}:{DENSE_LOSS_WEIGHT}",
    "AMP": config.amp,
})

## Finite-Loss And Gradient Sanity Check

Before launching DDP, one GPU builds the exact configured frozen teacher and random student. The check verifies a finite global+dense loss, non-zero student gradients, and no teacher gradients. The temporary module is deleted before distributed training starts.

In [ ]:
device = torch.device("cuda:0")
probe_detector = YOLO(YOLO_MODEL)
probe_encoder = YOLOBackboneEncoder(probe_detector.model).to(device)
probe_method = build_method(config, probe_encoder, device).to(device).train()
probe_images = torch.stack([preview_dataset[index] for index in range(2)]).to(device)
with torch.amp.autocast("cuda", enabled=config.amp):
    probe_loss = probe_method(probe_images)
assert torch.isfinite(probe_loss), f"Non-finite probe loss: {probe_loss.item()}"
probe_loss.backward()
student_grad_norm = sum(
    parameter.grad.detach().norm().item()
    for parameter in probe_method.online_encoder.parameters()
    if parameter.grad is not None
)
teacher_trainable = any(parameter.requires_grad for parameter in probe_method.teacher.parameters())
teacher_has_grad = any(parameter.grad is not None for parameter in probe_method.teacher.parameters())
assert student_grad_norm > 0, "Student encoder received no gradient."
assert not teacher_trainable and not teacher_has_grad, "Frozen DINOv3 teacher must not receive gradients."
pd.Series({
    "finite probe loss": float(probe_loss.detach()),
    "student encoder gradient norm": student_grad_norm,
    "teacher trainable": teacher_trainable, "teacher has gradients": teacher_has_grad,
})

del probe_method, probe_encoder, probe_detector, probe_images, probe_loss
torch.cuda.empty_cache()

## Distributed DINOv3-Guided Pretraining

The launcher writes the validated YAML and starts one process per T4 GPU. Each rank loads the same frozen DINOv3 teacher and a synchronized YOLOv12m student. The teacher is excluded from checkpoints; `best_ssl.pt` stores the student encoder, global projector, dense projector, optimizer, scheduler, and configuration.

The loss is feature alignment, not detection mAP. A lower loss means the CNN student matches this teacher more closely on AgroPest images; downstream 20%-label detection determines whether that transfer is useful.

In [ ]:
training_result = launch_distributed_pretrain(
    config, num_processes=runtime["cuda"]["device_count"],
    config_path=OUTPUT_DIR / "dinov3_config.yaml", check=True,
)
print(pd.Series({"completed": training_result.succeeded, "seconds": round(training_result.seconds, 1), "output": str(training_result.output_dir)}))

In [ ]:
# ============================================================
# Loss curve and detector-compatible checkpoint handoff
# ============================================================
history = pd.read_csv(OUTPUT_DIR / "history.csv")
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
LAST_SSL_CHECKPOINT = Path(manifest["outputs"]["ssl_checkpoint"])
SSL_CHECKPOINT = OUTPUT_DIR / "best_ssl.pt"
assert SSL_CHECKPOINT.is_file(), f"Missing lowest-loss checkpoint: {SSL_CHECKPOINT}"

fig, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", linewidth=2.2, ax=axis)
axis.set(title="DINOv3-Guided Global + Dense Distillation Loss", xlabel="Epoch", ylabel="Weighted cosine-regression loss")
axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "dinov3_distillation_loss.png", dpi=160); plt.show()

TRANSFERRED_YOLO = OUTPUT_DIR / "dinov3_best_backbone_yolo12m.pt"
transfer = transfer_ssl_backbone_to_yolo(
    SSL_CHECKPOINT, TRANSFERRED_YOLO, yolo_model=YOLO_MODEL, minimum_coverage=0.95,
)
pd.Series({
    "best SSL loss": manifest["best_loss"], "best SSL checkpoint": str(SSL_CHECKPOINT),
    "latest resumable checkpoint": str(LAST_SSL_CHECKPOINT),
    "trainer-exported student": manifest["outputs"]["yolo_checkpoint"],
    "detector-compatible best student": str(TRANSFERRED_YOLO),
    "backbone key coverage": f"{transfer.coverage:.1%}", "transfer report": str(transfer.report_json),
})

## Image-Only Representation Diagnostics

These diagnostics use a fixed image-only SSL-pool sample. They do not use validation/test images or labels. t-SNE is exploratory and preserves local neighborhoods only. The nearest-neighbor panels directly inspect whether the student embeds visually related pest, leaf, and scene structures near each other.

In [ ]:
diagnostic_paths = sorted(random.Random(SEED).sample(ssl_paths, min(DIAGNOSTIC_IMAGES, len(ssl_paths))))
evaluation_transform = v2.Compose([
    v2.Resize(IMAGE_SIZE + 32, antialias=True), v2.CenterCrop(IMAGE_SIZE), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
class SSLFeatureDataset(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return evaluation_transform(image.convert("RGB")), self.paths[index].name

feature_loader = DataLoader(SSLFeatureDataset(diagnostic_paths), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
diagnostic_detector = YOLO(str(TRANSFERRED_YOLO))
diagnostic_encoder = YOLOBackboneEncoder(diagnostic_detector.model).to(device).eval()
features, names = [], []
with torch.inference_mode():
    for images, batch_names in tqdm(feature_loader, desc="Extracting SSL-pool student features"):
        features.append(F.normalize(diagnostic_encoder(images.to(device, non_blocking=True)), dim=1).cpu())
        names.extend(batch_names)
feature_matrix = torch.cat(features)
assert torch.isfinite(feature_matrix).all()
pd.Series({
    "diagnostic images": len(names), "feature dimensions": feature_matrix.shape[1],
    "mean L2 norm": feature_matrix.norm(dim=1).mean().item(),
    "mean feature std": feature_matrix.std(dim=0).mean().item(),
})

In [ ]:
perplexity = min(30.0, max(5.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(n_components=2, perplexity=perplexity, learning_rate="auto", init="pca", max_iter=1000, random_state=SEED).fit_transform(feature_matrix.numpy())
fig, axis = plt.subplots(figsize=(9, 7))
axis.scatter(coordinates[:, 0], coordinates[:, 1], s=15, alpha=0.7, color="#d97706")
axis.set(title="t-SNE: DINOv3-Guided YOLOv12m SSL-Pool Features", xlabel="t-SNE 1", ylabel="t-SNE 2")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "dinov3_tsne_ssl_pool.png", dpi=160); plt.show()
pd.DataFrame({"filename": names, "tsne_1": coordinates[:, 0], "tsne_2": coordinates[:, 1]}).to_csv(OUTPUT_DIR / "ssl_pool_embeddings_tsne.csv", index=False)

query_indices = random.Random(SEED).sample(range(len(diagnostic_paths)), RETRIEVAL_QUERIES)
fig, axes = plt.subplots(RETRIEVAL_QUERIES, RETRIEVAL_NEIGHBORS + 1, figsize=(3.2 * (RETRIEVAL_NEIGHBORS + 1), 3.2 * RETRIEVAL_QUERIES))
retrieval_rows = []
for row, query_index in enumerate(query_indices):
    similarities = feature_matrix @ feature_matrix[query_index]
    neighbor_indices = [index for index in torch.topk(similarities, k=RETRIEVAL_NEIGHBORS + 1).indices.tolist() if index != query_index][:RETRIEVAL_NEIGHBORS]
    display_indices = [query_index] + neighbor_indices
    for column, index in enumerate(display_indices):
        with Image.open(diagnostic_paths[index]) as image: axes[row, column].imshow(image.convert("RGB"))
        title = "Query" if column == 0 else f"cos={similarities[index]:.3f}"
        axes[row, column].set_title(title, fontsize=9); axes[row, column].axis("off")
        retrieval_rows.append({"query": diagnostic_paths[query_index].name, "rank": column, "match": diagnostic_paths[index].name, "cosine_similarity": float(similarities[index])})
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "dinov3_nearest_neighbors.png", dpi=160); plt.show()
pd.DataFrame(retrieval_rows).to_csv(OUTPUT_DIR / "dinov3_nearest_neighbors.csv", index=False)

print("Interpretation to complete after execution: describe shared visual structure only, and defer detection-performance claims to Notebook 04b.")

## Checkpoint Publication And Viva Notes

Publish the complete `/kaggle/working/partB_04a_dinov3_yolo12m/` directory as a public Kaggle dataset. Notebook `04b` must attach it and use `best_ssl.pt` with `dinov3_best_backbone_yolo12m.pt` and its transfer report. The released DINOv3 teacher is not part of the downstream checkpoint.

- **Teacher:** released DINOv3 ViT-B/16, frozen throughout pretraining. It supplies global class-token and dense 14x14 patch-token targets.
- **Student:** randomly initialized YOLOv12m CNN. Its backbone is optimized to match the teacher, then transfers directly into the downstream YOLOv12m detector.
- **Global loss:** matches image-level semantic embeddings. **Dense loss:** matches spatial patch features, preserving structure relevant to detection. We use equal weights because both are mean cosine-regression losses and no YOLO-specific reweighting evidence exists.
- **Why not copy DINOv3 weights into YOLO?** ViT and YOLO CNN layers have incompatible tensor shapes and topology. Distillation transfers knowledge through feature targets rather than parameter copying.
- **Fairness:** this method receives an externally pretrained released teacher, so report it as domain-adaptive DINOv3-guided distillation rather than from-scratch SSL. SimCLR, BYOL, and I-JEPA start their YOLO students from random YAML architectures.
- **No label leakage:** both teacher and student receive only pixels from `ssl_pool/images`; labels, validation images, and test images are never read.
- **DINOv3 caveat:** this does not reproduce full DINOv3. There is no teacher EMA, iBOT, KoLeo, Gram anchoring, prototype assignment, or multi-crop self-distillation in this notebook.